In [1]:
# Cell 1
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.entity_resolution_service import (
    find_duplicate_candidates, link_organizations, list_organization_relations)
from src.ci.source_scoring import (
    score_source_artifact, batch_score_sources, get_source_score)
from src.engineering_intelligence.ip_mapping_service import (
    map_patent_to_claims, batch_map_all_patents, list_mappings_for_patent)
from src.engineering_intelligence.claim_conflict_service import (
    detect_conflicts, list_conflicts_for_construct)

driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
from config import get_driver
d = get_driver()
with d.session() as s:
    # 预声明所有 P1 新增属性键
    s.run("""
        MATCH (sa:SourceArtifact)
        WITH sa LIMIT 1
        SET sa.credibility_score = 0
        REMOVE sa.credibility_score
    """)
    s.run("""
        MATCH (o:Organization)
        WITH o LIMIT 1
        SET o.legal_name = ''
        REMOVE o.legal_name
    """)
    s.run("""
        MATCH (o:Organization)
        WITH o LIMIT 1
        SET o.website_domain = ''
        REMOVE o.website_domain
    """)
    s.run("""
        MATCH (pf:PatentFamily)
        WITH pf LIMIT 1
        SET pf.claim_scope_summary = ''
        REMOVE pf.claim_scope_summary
    """)
print("✅ 属性键预声明完成")

✅ 属性键预声明完成


In [3]:
# Cell 2: MCI-13 实体消歧
with driver.session() as s:
    target_org = s.run("""
        MATCH (o:Organization)
        RETURN o.organization_id AS oid, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 1
    """).single()

print(f"目标组织: {target_org['name']} ({target_org['oid']})")
candidates = find_duplicate_candidates(driver, target_org["oid"], min_score=0.3)
print(f"\n疑似重复候选: {len(candidates)} 条")
for c in candidates[:5]:
    print(f"  {c['candidate_canonical_name']}  score={c['score']}  reasons={c['reasons']}")

# 演示建立关系
with driver.session() as s:
    orgs = list(s.run("""
        MATCH (o:Organization)
        RETURN o.organization_id AS oid, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 2
    """))
if len(orgs) >= 2:
    try:
        rel_id = link_organizations(
            driver, orgs[0]["oid"], orgs[1]["oid"],
            relation_type="parent_of",
            note="演示关系",
        )
        print(f"✅ 建立关系: {rel_id}")
    except Exception as e:
        print(f"ℹ️ 建立关系跳过: {e}")

print("\n关系列表:")
for r in list_organization_relations(driver, orgs[0]["oid"]):
    print(f"  {r['source_name']} --[{r['relation_type']}]--> {r['target_name']}")

目标组织: Adaptive Phage Therapeutics (CI:ORG:8BC4DD44)

疑似重复候选: 0 条
✅ 建立关系: ORG-REL:CF05DF4D

关系列表:
  Adaptive Phage Therapeutics --[parent_of]--> Armata Pharmaceuticals


In [4]:
# Cell 3: MCI-14 来源可信度评分
with driver.session() as s:
    sid = s.run("MATCH (sa:SourceArtifact) RETURN sa.source_id AS sid LIMIT 1").single()["sid"]

print(f"目标来源: {sid}")
result = score_source_artifact(driver, sid)
print(json.dumps(result, ensure_ascii=False, indent=2))

print("\n批量评分:")
batch = batch_score_sources(driver, limit=500)
print(json.dumps(batch, ensure_ascii=False, indent=2))

print("\n读取评分:")
print(json.dumps(get_source_score(driver, sid), ensure_ascii=False, indent=2, default=str))

目标来源: SRC:a9d95a7467f8
{
  "score": 67,
  "tier": "secondary",
  "source_type": "press_release",
  "reasons": [
    "tier=secondary 基础分 55",
    "source_type=press_release 加分 +2",
    "存在 url +5",
    "存在 published_date +5"
  ],
  "source_id": "SRC:a9d95a7467f8"
}

批量评分:
{
  "total_candidates": 32,
  "scored": 32,
  "errors": []
}

读取评分:
{
  "source_id": "SRC:a9d95a7467f8",
  "credibility_score": 67,
  "credibility_tier": "secondary",
  "source_type": "press_release",
  "scored_at": "2026-09-30T09:03:28.293000000+00:00"
}


In [5]:
# Cell 4: EPI-11 专利-论文映射
with driver.session() as s:
    pf_row = s.run("""
        MATCH (pf:PatentFamily)
        WHERE pf.claim_scope_summary IS NOT NULL AND pf.claim_scope_summary <> ''
        RETURN pf.patent_family_id AS pfid, pf.title AS title LIMIT 1
    """).single()

if pf_row:
    print(f"目标专利族: {pf_row['title']} ({pf_row['pfid']})")
    r = map_patent_to_claims(driver, pf_row["pfid"], similarity_threshold=0.2, top_k=3)
    print(f"\n匹配到 {len(r['matches'])} 条 claim：")
    for m in r["matches"]:
        print(f"  sim={m['similarity']}  {m['claim_text'][:80]}")
    print(f"\n新建映射: {len(r['created_relations'])} 条")
else:
    print("⚠️ 没有带 claim_scope_summary 的专利族")

print("\n批量映射:")
batch = batch_map_all_patents(driver, similarity_threshold=0.2, top_k=3, limit=50)
print(json.dumps(batch, ensure_ascii=False, indent=2)[:1200])

⚠️ 没有带 claim_scope_summary 的专利族

批量映射:
{
  "total_patents_processed": 0,
  "total_mappings_created": 0,
  "details": []
}


In [6]:
# Cell 5: EPI-14 技术冲突检测
result = detect_conflicts(driver, persist=True)
print("=== 冲突检测结果 ===")
print(f"总冲突数: {result['total_conflicts']}")
print(f"  claim_status_conflict: {len(result['claim_status_conflicts'])}")
print(f"  result_direction_conflict: {len(result['result_direction_conflicts'])}")
print(f"  cross_construct_conflict: {len(result['cross_construct_conflicts'])}")
print(f"  created_relations: {len(result.get('created_relations', []))}")

for c in result["claim_status_conflicts"][:3]:
    print(f"\n  [{c['conflict_type']}] construct={c['construct_name']} claim_type={c['claim_type']}")
    print(f"    A: {c['claim_a_text']}")
    print(f"    B: {c['claim_b_text']}")
for c in result["result_direction_conflicts"][:3]:
    print(f"\n  [{c['conflict_type']}] construct={c['construct_name']} result_type={c['result_type']}")
    print(f"    A: {c['result_a_id']} / B: {c['result_b_id']}")
for c in result["cross_construct_conflicts"][:3]:
    print(f"\n  [{c['conflict_type']}] strategy={c['strategy_type']}")
    print(f"    A: {c['construct_a_name']} / B: {c['construct_b_name']}")

=== 冲突检测结果 ===
总冲突数: 0
  claim_status_conflict: 0
  result_direction_conflict: 0
  cross_construct_conflict: 0
  created_relations: 0


In [ ]:
from src.engineering_intelligence.claim_extractor import create_technical_claim, create_technical_result

with driver.session() as s:
    cid = s.run("""
        MATCH (ec:EngineeredPhageConstruct)
        RETURN ec.construct_id AS cid LIMIT 1
    """).single()["cid"]

# 造一条 contradicted 的 claim
create_technical_claim(
    driver,
    claim_type="host_range",
    claim_text="该构建体未能扩展宿主范围到 KL64 型",
    claimant_type="publication",
    evidence_context="in_vitro",
    construct_id=cid,
    actor_id="demo",
)
with driver.session() as s:
    s.run("""
        MATCH (tc:TechnicalClaim {claim_text: $t})
        SET tc.claim_status = 'contradicted'
    """, t="该构建体未能扩展宿主范围到 KL64 型")

In [7]:
# Cell 6: 全量汇总
print("=== 实体消歧 ===")
with driver.session() as s:
    rel_count = s.run("MATCH ()-[r:OWNS_OR_CONTROLS]->() RETURN count(r) AS c").single()["c"]
print(f"  OWNS_OR_CONTROLS 关系数: {rel_count}")

print("\n=== 来源评分 ===")
with driver.session() as s:
    scored = s.run("""
        MATCH (sa:SourceArtifact) WHERE sa.credibility_score IS NOT NULL
        RETURN count(sa) AS c, avg(sa.credibility_score) AS avg_score
    """).single()
print(f"  已评分: {scored['c']}, 平均分: {scored['avg_score']:.1f}")

print("\n=== 专利-论文映射 ===")
with driver.session() as s:
    m = s.run("MATCH ()-[r:MAPS_TO]->() RETURN count(r) AS c").single()["c"]
print(f"  MAPS_TO 关系数: {m}")

print("\n=== 技术冲突 ===")
with driver.session() as s:
    c = s.run("MATCH ()-[r:CONTRADICTS]->() RETURN count(r) AS c").single()["c"]
print(f"  CONTRADICTS 关系数: {c}")

=== 实体消歧 ===
  OWNS_OR_CONTROLS 关系数: 1

=== 来源评分 ===


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `MAPS_TO` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:MAPS_TO]->() RETURN count(r) AS c'


  已评分: 33, 平均分: 62.7

=== 专利-论文映射 ===


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `CONTRADICTS` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:CONTRADICTS]->() RETURN count(r) AS c'


  MAPS_TO 关系数: 0

=== 技术冲突 ===
  CONTRADICTS 关系数: 0
